In [2]:
#### Applying Roy's Coefficient to map from Landsat 7 to 8 ####
import pandas as pd
import os
import numpy as np

In [7]:
import pandas as pd

import os

csv_file = "/Volumes/Malonelab/Research/Whitezone/Data/KML/2013/2013_pts.csv"

df = pd.read_csv(csv_file)

# --------------------------------------------------
# Roy et al. (2016) harmonization
# Landsat 7 ETM+ -> Landsat 8 OLI
# --------------------------------------------------

# keep original Landsat 7 bands
B1 = df["SR_B2"]  # Blue
B2 = df["SR_B2"]  # Green
B3 = df["SR_B3"]  # Red
B4 = df["SR_B4"]  # NIR
B5 = df["SR_B5"]  # SWIR1
B7 = df["SR_B7"]  # SWIR2

# convert to Landsat 8 OLI equivalent

blue  = -0.0185 + 1.0053 * B1
green = -0.0129 + 1.0170 * B2
red   = -0.0140 + 1.0123 * B3
nir   = -0.0127 + 1.0126 * B4
swir1 = -0.0068 + 1.0043 * B5
swir2 = -0.0029 + 1.0000 * B7

# assign Landsat 8 band names
df["SR_B2"] = blue
df["SR_B3"] = green
df["SR_B4"] = red
df["SR_B5"] = nir
df["SR_B6"] = swir1
df["SR_B7"] = swir2

# --------------------------------------------------
# Recalculate vegetation indices
# --------------------------------------------------

# NDVI
df["NDVI"] = (nir - red) / (nir + red)

# NDMI
df["NDMI"] = (nir - swir1) / (nir + swir1)

# SAVI (L = 0.5)
df["SAVI"] = 1.5 * (nir - red) / (nir + red + 0.5)

# Clean any divide-by-zero results
for col in ["NDVI", "NDMI", "SAVI"]:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)

# --------------------------------------------------
# Save
# --------------------------------------------------
out_file = os.path.join(
    os.path.dirname(csv_file),
    "2013_pts_Roy.csv"
)

df.to_csv(out_file, index=False)

print(f"Saved: {out_file}")
print(df[["NDVI", "NDMI", "SAVI"]].describe())

Saved: /Volumes/Malonelab/Research/Whitezone/Data/KML/2013/2013_pts_Roy.csv
                NDVI           NDMI           SAVI
count  558391.000000  558398.000000  558391.000000
mean       -0.004773      -0.142441      -0.007159
std         0.017426       0.120898       0.026138
min        -0.427481      -0.884842      -0.641087
25%        -0.016299      -0.238275      -0.024448
50%        -0.004445      -0.145805      -0.006668
75%         0.007251      -0.052855       0.010876
max         0.859563       0.280988       1.289033
